# Milestone 5: End-to-end demo and your own scenario
**Project: Northwind Knowledge Assistant** | **Presenter: Nagarjuna Naidu** | Day 5, remaining time

Goal: see the whole assistant work on five questions, check what this build is made of, then adapt it to your own scenario.

In [ ]:
# Optional: connect a free LLM. Remove the leading "# ", fill in your key, then restart the kernel.
# import os
# os.environ["LLM_API_KEY"] = "your-key"
# os.environ["LLM_BASE_URL"] = "https://api.groq.com/openai/v1"   # Google AI Studio: https://generativelanguage.googleapis.com/v1beta/openai/ ; Ollama: http://localhost:11434/v1
# os.environ["LLM_MODEL"] = "llama-3.3-70b-versatile"             # any chat model your provider offers
# Azure OpenAI instead (use these four lines INSTEAD of the three lines above; the key is given in class, never save it in a file you share):
# os.environ["AZURE_OPENAI_ENDPOINT"] = "https://<your-resource>.openai.azure.com/"
# os.environ["AZURE_OPENAI_API_KEY"] = "your-azure-key"
# os.environ["AZURE_OPENAI_API_VERSION"] = "2024-10-21"
# os.environ["LLM_MODEL"] = "gpt-4.1-mini"   # the DEPLOYMENT name in Azure
# No key? Leave this cell as it is: everything runs in offline demo mode.

## 1. Five questions, end to end
Retrieval with role-based access, versioned policies, sources, refusals, and an agent for the multi-step task.

In [ ]:
from nka.security import secure_answer
from nka.agent import run_agent
from nka.llm_client import llm_mode

print(llm_mode(), "\n")

def show(question, role="employee"):
    r = secure_answer(question, role=role)
    print(f"[{role}] {question}\n  A: {r['answer'][:230]}")
    print("  Sources:", "; ".join(dict.fromkeys(s.split("] ", 1)[1] for s in r["sources"])) or "none", "\n")

show("Can I work from Spain for a month?")
show("How many weeks of parental leave do I get?")                  # newest policy version only
show("Which vendors are under enhanced monitoring?", "intern")      # confidential: refused
show("Which vendors are under enhanced monitoring?", "hr_partner")  # same question, more access
run_agent("Which London bookings exceed the hotel cap in our travel policy, and what is the total overage?")

## 2. Architecture facts of THIS build
Read from the code and configuration, not typed by hand.

In [ ]:
import os
from nka import config, security
from nka.data import load_docs
from nka.index import get_index
from nka.tools import TOOLS

print("Embedding model :", config.MODEL_NAME, "| LLM:", llm_mode().split("(")[0].strip())
print("Retrieval       :", len(load_docs()), "docs,", len(get_index().chunks), "chunks of 150 words, cosine top-3, refuse below", config.MIN_SIM)
print("Access control  :", {r: len(v) for r, v in config.ROLES.items()}, "sensitivity levels per role; superseded versions hidden")
print("Agent tools     :", ", ".join(TOOLS), "| allow-list, step limit 5, errors caught | MCP server 'nka' exposes the same registry")
print("Tracking        :", "MLflow (sqlite)" if (config.ARTIFACTS / "mlflow.db").exists() else "MLflow (run M3 first)", "| best config:", (config.ARTIFACTS / "best_config.json").exists())
print("Monitoring      :", "PSI per feature (0.1 / 0.25), thumbs-up A/B test, query log CSV")
print("Security        :", len(security.PATTERNS), "injection patterns, PII masking, <data> hardened prompt, canary output check")

## 3. Adapt it to your scenario
**A** bank | **B** insurer | **C** retail / telecom | **D** your own. Keep the same skeleton: documents, roles, tools, evaluation, monitoring, controls.

**Canvas checklist (9 sections):** 1 problem and users, 2 data sources, 3 architecture, 4 model choices, 5 tools and agents, 6 evaluation and monitoring, 7 deployment and cost, 8 risk, security and fairness, 9 roadmap and value.

**8-minute presentation:** problem (1 min), architecture (2), live demo (2), evaluation and operations (1), risks and controls (1), next steps (1).

| Criterion | Points |
|---|---|
| Fit to the scenario | 20 |
| Architecture | 25 |
| Data and governance | 15 |
| Evaluation and operations | 15 |
| Responsible AI, security, compliance | 15 |
| Clarity | 10 |

## 4. YOUR TASK
Change the role and the questions to fit your scenario (your own documents go in `data/corpus`), then rerun.

In [ ]:
role = "employee"                                  # TODO: your role: intern, employee or hr_partner
questions = ["How do I reset my MFA?"]             # TODO: 2 or 3 questions from your scenario
for q in questions:
    show(q, role)

**Canvas, all sections:** write one line in each of the 9 sections, then draft your 8-minute story.